# Group3 Assignment 1: Data Cleaning

In [30]:
import pandas as pd
import numpy as np
from scipy.stats import norm

transaction_data = pd.read_csv('../data/transaction_data.csv')


# From Assignment 1

## Question 1 - Dataframe formatting

##### As you load and inspect your transaction dataframe, you can observe that the column names do not follow a similar convention. Apply the changes necessary to ensure all column names follow the same convention.

In [31]:
transaction_data.head()

,sku_number,inventory_type,stocking_type,leadTime,unit_price,Manufacturing Site,division_code,transaction_date,Order-Quantity
0,F544EBC2,WIP,MTS,28,993.199814,NaN,514,1/1/2023,97
1,F6D696A7,FG,MTS,28,1011.724796,NaN,2056,1/1/2023,95
2,53B542CB,FG,MTS,28,1005.158694,PL-5C,06CC,1/1/2023,103
3,FE55EA7C,FG,MTS,28,997.138166,NaN,02B5,1/1/2023,58
4,BF7C4C4D,FG,MTO,28,998.995060,US-86,BCF8,1/1/2023,93


leadTime, Manufacturing Site, and Order-Quantity headings don't match the word_word format used in the other columns so those will get converted to the word_word format to be consistent.

In [32]:
transaction_data = transaction_data.rename(columns={"leadTime":"lead_time","Manufacturing Site":"manufacturing_site","Order-Quantity":"order_quantity"})
transaction_data.head()

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
0,F544EBC2,WIP,MTS,28,993.199814,NaN,514,1/1/2023,97
1,F6D696A7,FG,MTS,28,1011.724796,NaN,2056,1/1/2023,95
2,53B542CB,FG,MTS,28,1005.158694,PL-5C,06CC,1/1/2023,103
3,FE55EA7C,FG,MTS,28,997.138166,NaN,02B5,1/1/2023,58
4,BF7C4C4D,FG,MTO,28,998.995060,US-86,BCF8,1/1/2023,93


Column names corrected!

# Question 2 — Data Inspection

##### Inspect the numeric features of your dataframe with the describe function. Provide a summary of these features, and in case there are any oddities/anomalies in your dataframe, locate and correct them.  Please remember to provide a note on any oddities you have found and a rationale for your suggested correction. You can do so by including a text for the changes you have applied in a corresponding markdown shell.

In [33]:
transaction_data.describe()

,lead_time,unit_price,order_quantity
count,400634.000000,400634.000000,400634.000000
mean,25.760976,800.308766,79.933211
std,6.465948,374.746323,41.930024
min,14.000000,-85.222467,-98.000000
25%,28.000000,982.549477,63.000000
50%,28.000000,998.073561,93.000000
75%,28.000000,1006.155114,109.000000
max,2800.000000,1023.638643,194.000000


There is a max leadtime of 2800 which seems strange.  There are negative unit price and order quantity values which could be odd except that these are for returns or some type of reverse transaction.

In [34]:
maxLT = transaction_data[transaction_data["lead_time"] == 2800]
maxLT

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
211554,E1A661D9,FG,MTS,2800,1018.133363,JP-08,EBDE,1/21/2024,72


There is only 1 row where the lead time in 2800.  Below I create a list of other rows with the same sku to see if there is a standard lead time for this part like 28 so that we can see if the 2800 can be corrected.

In [35]:
E1A661D9 = transaction_data[transaction_data["sku_number"] == "E1A661D9"]
E1A661D9.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2968 entries, 144 to 400601
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   sku_number          2968 non-null   object 
 1   inventory_type      2968 non-null   object 
 2   stocking_type       2968 non-null   object 
 3   lead_time           2968 non-null   int64  
 4   unit_price          2968 non-null   float64
 5   manufacturing_site  2288 non-null   object 
 6   division_code       2785 non-null   object 
 7   transaction_date    2968 non-null   object 
 8   order_quantity      2968 non-null   int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 231.9+ KB


In [36]:
E1A661D9.head()

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
144,E1A661D9,FG,MTS,28,1018.133363,PL-D8,2DEB,1/1/2023,73
390,E1A661D9,FG,MTS,21,200.187903,PL-D8,1D32,1/1/2023,46
597,E1A661D9,FG,MTS,28,1018.133363,PL-F5,4DC6,1/2/2023,91
688,E1A661D9,FG,ATO,14,45.719821,NaN,9B08,1/2/2023,-6
716,E1A661D9,FG,MTS,28,1018.133363,PL-D8,629D,1/2/2023,108


Nope. This sku has different lead times in different scenarios so I will remove the row with the lead time of 2800.

In [37]:
transaction_data = transaction_data[transaction_data["lead_time"] != 2800]
transaction_data.describe()

,lead_time,unit_price,order_quantity
count,400633.000000,400633.000000,400633.000000
mean,25.754052,800.308222,79.933231
std,4.753723,374.746633,41.930075
min,14.000000,-85.222467,-98.000000
25%,28.000000,982.549477,63.000000
50%,28.000000,998.073561,93.000000
75%,28.000000,1006.155114,109.000000
max,28.000000,1023.638643,194.000000


lead_time = 2800 removed!

# Question 3 — NaN Values

##### Your transactional dataset includes many missing values. Find and address (in your dataframe) the missing values by column. Once more, remember to provide a note on the NaN values you found and a rationale for your suggested action.

In [38]:
transaction_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 400633 entries, 0 to 400633
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_number          399368 non-null  object 
 1   inventory_type      394721 non-null  object 
 2   stocking_type       394192 non-null  object 
 3   lead_time           400633 non-null  int64  
 4   unit_price          400633 non-null  float64
 5   manufacturing_site  300563 non-null  object 
 6   division_code       376876 non-null  object 
 7   transaction_date    400633 non-null  object 
 8   order_quantity      400633 non-null  int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 30.6+ MB


The total number of rows is 400,633 so there are null values in columns sku_number, inventory_type, stocking_type, manufacturing_site, and division_code.  Since all of these columns seem critical to being able to track inventory usage by location (which is what I think I will need to use this data for calculating safety stock later) I don't think I can use any rows that have a null value.

Therefore I am going to remove any rows with null values.

In [39]:
transaction_data_clean = transaction_data.dropna()
transaction_data_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 272975 entries, 2 to 400631
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_number          272975 non-null  object 
 1   inventory_type      272975 non-null  object 
 2   stocking_type       272975 non-null  object 
 3   lead_time           272975 non-null  int64  
 4   unit_price          272975 non-null  float64
 5   manufacturing_site  272975 non-null  object 
 6   division_code       272975 non-null  object 
 7   transaction_date    272975 non-null  object 
 8   order_quantity      272975 non-null  int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 20.8+ MB


Null values removed!

# Question 4 — Useful Information

##### It's time to focus on the subset of this dataset we need. Since we are ultimately interested in calculating safety stock, we would like to focus on finished goods that are designated as "make to stock." Create a filtered subset of your transaction dataset and write appropriate code to answer the following questions (for the filtered data).
- How many unique SKUs are we working with?
- How many unique Manufacturing Sites are we working with?
- How many Divisions are we working with?
- What are the top and bottom 10 transactions by order quantity?
- What are the top and bottom 10 transactions by total sales value? Hint: Notice sales is not a feature of this dataset, thus we must calculate it using the existing features.


In [40]:
sku_count = transaction_data_clean["sku_number"].nunique()
sku_count

480

There are 480 unique SKUs

In [41]:
site_count = transaction_data_clean["manufacturing_site"].nunique()
site_count

15

There are 15 unique manufacturing sites

In [42]:
division_count = transaction_data_clean["division_code"].nunique()
division_count

66

There are 66 unique divisions

In [43]:
top_order_quantity = transaction_data_clean.nlargest(n = 10, columns = "order_quantity")
bottom_order_quantity = transaction_data_clean.nsmallest(n = 10, columns = "order_quantity")

Below are the top 10 records by order quantity

In [44]:
top_order_quantity

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
191625,AD8D5363,WIP,MTS,28,1001.452540,PL-F5,514,12/15/2023,193
77179,D5C05A4C,FG,MTO,28,990.594665,PL-F5,C40B,5/21/2023,184
350090,6374C42B,RM,MTS,28,1009.715112,PL-D8,2DEB,9/30/2024,182
351878,32843A68,FG,MTS,28,1007.025152,US-86,06CC,10/3/2024,182
357371,73803309,RM,MTS,28,1018.136250,JP-2B,3B87,10/13/2024,182
305472,0336A033,FG,MTS,28,999.224537,JP-8C,EBDE,7/10/2024,181
56975,0019425F,FG,MTS,28,1004.366509,US-86,06CC,4/14/2023,180
155580,6970C929,FG,MTO,28,998.891440,JP-F8,3B87,10/11/2023,180
254637,7D60C5D8,FG,MTS,28,997.172751,PL-D8,7F24,4/8/2024,180
276850,633E44A4,FG,MTS,28,1001.723540,PL-F5,EBDE,5/19/2024,180


Below are the bottom 10 records by order quantity

In [45]:
bottom_order_quantity

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
300449,79FAFFF1,WIP,MTS,21,207.344912,CA-0B,E202,7/1/2024,-54
36123,BC55FF7F,FG,MTO,21,191.327975,PL-E4,02B5,3/7/2023,-52
276913,0980E63F,RM,MTO,21,200.425909,PL-4A,514,5/19/2024,-50
312078,73803309,FG,MTO,21,218.328503,PL-D8,2056,7/22/2024,-50
274486,313AF9CD,FG,MTO,21,194.559157,PL-F5,EFE6,5/15/2024,-49
107807,BED0B828,RM,MTS,21,205.361776,PL-D8,514,7/16/2023,-48
224230,C477701E,FG,MTO,21,202.881912,PL-F5,C40B,2/13/2024,-46
226265,9115DE13,FG,MTO,21,204.428461,PL-F5,C672,2/17/2024,-46
377803,CBA11DC3,FG,MTS,21,206.590119,JP-55,C672,11/19/2024,-45
53756,BCB42CB2,FG,MTS,21,220.316820,PL-D8,629D,4/8/2023,-44


Now creating a new total_sales column

In [46]:
transaction_data_clean["total_sales"] = transaction_data_clean["unit_price"] * transaction_data_clean["order_quantity"]

C:\Users\gibbs002\AppData\Local\Temp\ipykernel_15448\1943276211.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transaction_data_clean["total_sales"] = transaction_data_clean["unit_price"] * transaction_data_clean["order_quantity"]


In [47]:
transaction_data_clean.head()

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,total_sales
2,53B542CB,FG,MTS,28,1005.158694,PL-5C,06CC,1/1/2023,103,103531.345482
4,BF7C4C4D,FG,MTO,28,998.995060,US-86,BCF8,1/1/2023,93,92906.540589
6,3D7DF103,FG,MTS,28,996.426246,PL-F5,3B87,1/1/2023,104,103628.329584
9,E1CC60BC,FG,MTO,28,997.554097,PL-F5,BCF8,1/1/2023,92,91774.976915
10,740A6465,FG,MTO,28,995.922511,CA-0B,514,1/1/2023,121,120506.623831


In [48]:
top_total_sales = transaction_data_clean.nlargest(n = 10, columns = "total_sales")
bottom_total_sales = transaction_data_clean.nsmallest(n = 10, columns = "total_sales")

Below are the top 10 records by total sales value

In [49]:
top_total_sales

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,total_sales
191625,AD8D5363,WIP,MTS,28,1001.452540,PL-F5,514,12/15/2023,193,193280.340220
357371,73803309,RM,MTS,28,1018.136250,JP-2B,3B87,10/13/2024,182,185300.797500
350090,6374C42B,RM,MTS,28,1009.715112,PL-D8,2DEB,9/30/2024,182,183768.150384
351878,32843A68,FG,MTS,28,1007.025152,US-86,06CC,10/3/2024,182,183278.577664
77179,D5C05A4C,FG,MTO,28,990.594665,PL-F5,C40B,5/21/2023,184,182269.418360
72168,2BC32D8E,FG,MTO,28,1010.985767,PL-F5,2056,5/12/2023,179,180966.452293
305472,0336A033,FG,MTS,28,999.224537,JP-8C,EBDE,7/10/2024,181,180859.641233
56975,0019425F,FG,MTS,28,1004.366509,US-86,06CC,4/14/2023,180,180785.971620
276850,633E44A4,FG,MTS,28,1001.723540,PL-F5,EBDE,5/19/2024,180,180310.237200
155580,6970C929,FG,MTO,28,998.891440,JP-F8,3B87,10/11/2023,180,179800.459128


Below are the bottom 10 records by total sales value

In [50]:
bottom_total_sales

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,total_sales
300449,79FAFFF1,WIP,MTS,21,207.344912,CA-0B,E202,7/1/2024,-54,-11196.625237
312078,73803309,FG,MTO,21,218.328503,PL-D8,2056,7/22/2024,-50,-10916.425135
276913,0980E63F,RM,MTO,21,200.425909,PL-4A,514,5/19/2024,-50,-10021.295475
36123,BC55FF7F,FG,MTO,21,191.327975,PL-E4,02B5,3/7/2023,-52,-9949.054705
107807,BED0B828,RM,MTS,21,205.361776,PL-D8,514,7/16/2023,-48,-9857.365253
53756,BCB42CB2,FG,MTS,21,220.316820,PL-D8,629D,4/8/2023,-44,-9693.940071
274486,313AF9CD,FG,MTO,21,194.559157,PL-F5,EFE6,5/15/2024,-49,-9533.398678
226265,9115DE13,FG,MTO,21,204.428461,PL-F5,C672,2/17/2024,-46,-9403.709220
224230,C477701E,FG,MTO,21,202.881912,PL-F5,C40B,2/13/2024,-46,-9332.567952
377803,CBA11DC3,FG,MTS,21,206.590119,JP-55,C672,11/19/2024,-45,-9296.555346


# From Assignment 2

In [58]:
df_agg = transaction_data_clean.groupby('sku_number').agg(
    # Calculate min, max, mean, median, variance, and std dev for order quantity
    qty_min=('order_quantity', 'min'),
    qty_max=('order_quantity', 'max'),
    qty_avg=('order_quantity', 'mean'),
    qty_median=('order_quantity', 'median'),
    qty_var=('order_quantity', 'var'),
    qty_std=('order_quantity', 'std'),
    
    avg_lead_time=('lead_time', 'mean')
).reset_index()
# Display the first 5 rows of the new aggregated DataFrame
print("\nAggregated SKU Statistics (Head):")
df_agg.head()


Aggregated SKU Statistics (Head):


,sku_number,qty_min,qty_max,qty_avg,qty_median,qty_var,qty_std,avg_lead_time
0,0019425F,-23,180,94.795109,98.0,762.127076,27.606649,27.532065
1,00DCA10C,-25,29,6.116732,6.0,99.041008,9.951935,14.000000
2,010BA6D0,-15,33,3.659091,4.0,107.346195,10.360801,14.000000
3,01FE860A,-25,70,15.840637,13.0,343.262502,18.527345,18.629482
4,022AED39,-20,167,96.156317,99.0,786.451181,28.043737,27.370450


Question 2 is featured below, defining service levels and getting the z-scores necessary for safety stock calculations.

In [59]:
# Define the target cycle service levels (CSL) as probabilities
service_levels = [0.75, 0.90, 0.95]
z_scores = {sl: norm.ppf(sl) for sl in service_levels}

# Print the calculated Z-scores formatted to 4 decimal places
print(f"Z-score for 75% SL: {z_scores[0.75]:.4f}")
print(f"Z-score for 90% SL: {z_scores[0.90]:.4f}")
print(f"Z-score for 95% SL: {z_scores[0.95]:.4f}")

Z-score for 75% SL: 0.6745
Z-score for 90% SL: 1.2816
Z-score for 95% SL: 1.6449


The sqrt(L) factor gets used to get the last lead time for the SS to calculate with, which is necessary to get our final results for the 3 different requirements/z-scores.

In [60]:
df_ss = df_agg.copy()

df_ss['sqrt_lead_time'] = np.sqrt(df_ss['avg_lead_time'])
# Calculating the raw safety stock based on the formula SS = Z * std_dev_demand_per_period * sqrt(lead_time_in_periods)
df_ss['ss_75_raw'] = z_scores[0.75] * df_ss['qty_std'] * df_ss['sqrt_lead_time']
df_ss['ss_90_raw'] = z_scores[0.90] * df_ss['qty_std'] * df_ss['sqrt_lead_time']
df_ss['ss_95_raw'] = z_scores[0.95] * df_ss['qty_std'] * df_ss['sqrt_lead_time']

df_ss['ss_75'] = np.ceil(df_ss['ss_75_raw']).fillna(0)
df_ss['ss_90'] = np.ceil(df_ss['ss_90_raw']).fillna(0)
df_ss['ss_95'] = np.ceil(df_ss['ss_95_raw']).fillna(0)
# Display the final safety stock values for the first 5 SKUs
print("SKU Safety Stock Calculations (Head):")
df_ss[['sku_number', 'ss_75', 'ss_90', 'ss_95']].head()

SKU Safety Stock Calculations (Head):


,sku_number,ss_75,ss_90,ss_95
0,0019425F,98.0,186.0,239.0
1,00DCA10C,26.0,48.0,62.0
2,010BA6D0,27.0,50.0,64.0
3,01FE860A,54.0,103.0,132.0
4,022AED39,99.0,189.0,242.0


We use nlargest().

In [61]:
# Find the SKU with the single largest safety stock at the 95% service level
sku_largest_ss = df_ss.nlargest(1, 'ss_95')
print("SKU with the largest safety stock (95% SL):")
sku_largest_ss[['sku_number', 'ss_95']]

SKU with the largest safety stock (95% SL):


,sku_number,ss_95
231,79FAFFF1,375.0


We use nsmallest(). Note that a value of 0 is possible if an SKU had 0 variance or only one transaction.

In [64]:
# Find the SKU with the single smallest safety stock at the 95% service level
sku_smallest_ss = df_ss.nsmallest(1, 'ss_95')
print("SKU with the smallest safety stock (95% SL):")
sku_smallest_ss[['sku_number', 'ss_95']]

SKU with the smallest safety stock (95% SL):


,sku_number,ss_95
341,B1476889,42.0


In [65]:
avg_ss_95 = df_ss['ss_95'].mean()
print(f"The average safety stock (95% SL) across all SKUs is: {avg_ss_95:.2f} units")

The average safety stock (95% SL) across all SKUs is: 148.96 units


The side-quest is featured below, starting with mstats being imported. cal_mquantiles, SKU order quantity calculations, and non-parametric safety stock calculations will follow.

In [66]:
# Import mstats (masked statistics) from scipy.stats
from scipy.stats import mstats

print("Imported scipy.stats.mstats for non-parametric calculations.")

Imported scipy.stats.mstats for non-parametric calculations.


This function below uses scipy.stats.mstats to calculate the 75th, 90th, and 95th empirical quantiles for a data series. It's designed to work perfectly with pandas' .apply() method by returning a new series

In [67]:
# Define a helper function that we can use with pandas' .apply() method
def calc_mquantiles(series):
    """
    Calculates the 75th, 90th, and 95th empirical quantiles
    for a pandas series using mstats.
    """
    quantiles = mstats.mquantiles(series, prob=[0.75, 0.90, 0.95])
    
    return pd.Series(quantiles, index=[0.75, 0.90, 0.95])

print("Helper function calc_mquantiles defined.")

Helper function calc_mquantiles defined.


Group by SKU, apply the calc_mquantiles helper function to find the 75th, 90th, and 95th percentile values from the actual data, and then rename the columns for clarity.

In [70]:
print("Calculating empirical quantiles for each SKU's order_quantity...")

df_quantiles_np = transaction_data_clean.groupby('sku_number')['order_quantity'].apply(calc_mquantiles).unstack()
# Rename the columns from the probability (e.g., 0.75) to a descriptive name (e.g., q_np_75)
df_quantiles_np = df_quantiles_np.rename(columns={
    0.75: 'q_np_75',
    0.90: 'q_np_90',
    0.95: 'q_np_95'
})

print("Quantile calculation complete. Head of quantiles:")
df_quantiles_np.head()

Calculating empirical quantiles for each SKU's order_quantity...
Quantile calculation complete. Head of quantiles:


,q_np_75,q_np_90,q_np_95
sku_number,,,
0019425F,112.00,124.00,131.00
00DCA10C,14.00,18.00,22.00
010BA6D0,10.55,16.18,19.73
01FE860A,27.00,42.48,52.00
022AED39,113.00,126.00,132.19


Computes the final non-parametric safety stock using the formula SS = (Quantile - Average) * sqrt(Lead Time), ensuring the result is non-negative and rounded up.

In [71]:
# Merge the original aggregated stats (df_agg) with the new non-parametric quantiles (df_quantiles_np)
df_ss_np = df_agg.merge(df_quantiles_np, on='sku_number')


df_ss_np['sqrt_lead_time'] = np.sqrt(df_ss_np['avg_lead_time'])

base_ss_75 = (df_ss_np['q_np_75'] - df_ss_np['qty_avg']).clip(lower=0)
base_ss_90 = (df_ss_np['q_np_90'] - df_ss_np['qty_avg']).clip(lower=0)
base_ss_95 = (df_ss_np['q_np_95'] - df_ss_np['qty_avg']).clip(lower=0)

df_ss_np['ss_np_75_raw'] = base_ss_75 * df_ss_np['sqrt_lead_time']
df_ss_np['ss_np_90_raw'] = base_ss_90 * df_ss_np['sqrt_lead_time']
df_ss_np['ss_np_95_raw'] = base_ss_95 * df_ss_np['sqrt_lead_time']

df_ss_np['ss_np_75'] = np.ceil(df_ss_np['ss_np_75_raw']).fillna(0)
df_ss_np['ss_np_90'] = np.ceil(df_ss_np['ss_np_90_raw']).fillna(0)
df_ss_np['ss_np_95'] = np.ceil(df_ss_np['ss_np_95_raw']).fillna(0)
# Display the final non-parametric safety stock values
print("\nNon-Parametric Safety Stock Calculations (Head):")
df_ss_np[['sku_number', 'ss_np_75', 'ss_np_90', 'ss_np_95']].head()


Non-Parametric Safety Stock Calculations (Head):


,sku_number,ss_np_75,ss_np_90,ss_np_95
0,0019425F,91.0,154.0,190.0
1,00DCA10C,30.0,45.0,60.0
2,010BA6D0,26.0,47.0,61.0
3,01FE860A,49.0,115.0,157.0
4,022AED39,89.0,157.0,189.0
